# Physics‑Informed Fourier Neural Operator (PI‑FNO) 
## Coupled 3D Thermo‑Mechanical Box (Hexahedral FEM)

This notebook demonstrates **physics‑informed operator learning** for a coupled thermo‑mechanical problem in 3D. The objective is to learn an operator that maps a **parameter field** (here: a spatially varying conductivity‑like field $K$) to the coupled state fields:
$$
K \;\mapsto\; \left(T,\; U_x,\; U_y,\; U_z\right)
$$
without relying on paired finite‑element solution labels during training.

### Workflow (high‑level)
1. Construct a structured **3D hexahedral FE mesh** of a unit box.
2. Define a **monolithic coupled thermo‑mechanical residual** evaluated at element level.
3. Generate randomized material fields $K$ using a **Fourier parameterization**.
4. Build a **multi‑output Fourier Neural Operator (FNO)** surrogate for $(T,U_x,U_y,U_z)$.
5. Train the surrogate by minimizing the **physics residual loss** (physics‑informed learning).
6. Validate on selected samples by comparing against a **nonlinear FE solve**.
7. Export fields (state + derived quantities) for downstream visualization.



## 1) Directory handling and logging

We create a clean working directory and redirect `stdout` into a log file, so training progress is saved.


In [1]:
import os
from fol.tools.usefull_functions import *
# directory & save handling
working_directory_name = 'pi_fno_thermo_mechanical_3d_box'
case_dir = os.path.join('.', working_directory_name)
create_clean_directory(working_directory_name)


## 2) Problem / model settings

`L` and `N` define the box geometry and mesh resolution.

Boundary conditions are set for:
- Temperature `T`
- Displacements `Ux, Uy, Uz`

Here, left and right faces are prescribed with different values.
Then, We build a structured 3D box mesh and initialize it.

In [2]:
model_settings = {
    "L": 1,
    "N": 10,  # number of elements per direction (mesh will have N+1 nodes per axis)

    # Displacement BCs
    "Ux_left": 0.0, "Ux_right": 0.10,
    "Uy_left": 0.0, "Uy_right": 0.10,
    "Uz_left": 0.0, "Uz_right": 0.10,

    # Temperature BCs
    "T_left": 0.5, "T_right": 0.0
}

from fol.tools.usefull_functions import create_3D_box_mesh
# creation of the mesh
fe_mesh = create_3D_box_mesh(
    Nx=model_settings["N"],
    Ny=model_settings["N"],
    Nz=model_settings["N"],
    Lx=model_settings["L"],
    Ly=model_settings["L"],
    Lz=model_settings["L"],
    case_dir=case_dir
)
fe_mesh.Initialize()


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Extruded)
Info    : [ 10%] Meshing curve 2 (Extruded)
Info    : [ 20%] Meshing curve 3 (Extruded)
Info    : [ 30%] Meshing curve 4 (Extruded)
Info    : [ 40%] Meshing curve 7 (Extruded)
Info    : [ 50%] Meshing curve 8 (Extruded)
Info    : [ 60%] Meshing curve 9 (Extruded)
Info    : [ 60%] Meshing curve 10 (Extruded)
Info    : [ 70%] Meshing curve 12 (Extruded)
Info    : [ 80%] Meshing curve 13 (Extruded)
Info    : [ 90%] Meshing curve 17 (Extruded)
Info    : [100%] Meshing curve 21 (Extruded)
Info    : Done meshing 1D (Wall 7.98795e-05s, CPU 5.3e-05s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 5 (Extruded)
Info    : [ 20%] Meshing surface 14 (Extruded)
Info    : [ 40%] Meshing surface 18 (Extruded)
Info    : [ 60%] Meshing surface 22 (Extruded)
Info    : [ 70%] Meshing surface 26 (Extruded)
Info    : [ 90%] Meshing surface 27 (Extruded)
Info    : Done meshing 2D (Wall 0.00202655s, CPU 0.002684s)
Info    : Meshing 

## 3) Define the thermo-mechanical physics loss

We configure:
- **Dirichlet boundary conditions** for `T, Ux, Uy, Uz`
- **Material properties** (elasticity + reference temperature)
- **Thermal properties** (`alpha`, `beta`, `c`)

The `ThermoMechanicsLoss3DHexa` object is responsible for evaluating PDE residuals / energy terms for training.


## Thermo‑Mechanical Finite‑Element Residual Loss

### Scientific description
The loss evaluates a **monolithically coupled thermo‑mechanical finite‑element residual** in 3D, with unknowns given by the temperature field $T$ and the displacement vector $\mathbf{u}=(U_x,U_y,U_z)$. Training minimizes the violation of the governing PDEs over all elements, enabling **physics‑informed learning** without supervised FE labels.

The formulation is consistent with:

**(i) Steady heat conduction**
$$
\nabla\cdot\left(k\,\nabla T\right)=0,
$$
where $k$ is a spatially varying conductivity‑type coefficient (here parameterized as $K$).

**(ii) Linear momentum balance**
$$
\nabla\cdot\boldsymbol{\sigma}+\mathbf{b}=0,
$$
with linear‑elastic constitutive response and thermal expansion coupling:
$$
\boldsymbol{\sigma}=\mathbf{D}\left(\boldsymbol{\varepsilon}-\boldsymbol{\varepsilon}_\text{th}\right),
\qquad
\boldsymbol{\varepsilon}_\text{th}=\alpha\,(T-T_0)\,\mathbf{s}.
$$
Here $\mathbf{D}$ is the elastic constitutive matrix, $\alpha$ is the thermal expansion coefficient, $T_0$ is a reference temperature field, and $\mathbf{s}$ is a dimension‑appropriate selector defining the thermal strain components.

### What the loss returns computationally
At the element level, the loss implementation computes:
- a **total scalar loss** (weighted-residual objective) for efficient gradient‑based learning,
- the **coupled residual vector**,
- and **coupled Jacobian (tangent) matrix** required for Newton‑type FE solves.



In [3]:
from fol.loss_functions.thermo_mechanics import ThermoMechanicsLoss3DHexa
import numpy as np

# Boundary condition dictionary (Dirichlet-style values on left/right faces)
bc_dict = {
    "T":  {"left": model_settings["T_left"],  "right": model_settings["T_right"]},
    "Ux": {"left": model_settings["Ux_left"], "right": model_settings["Ux_right"]},
    "Uy": {"left": model_settings["Uy_left"], "right": model_settings["Uy_right"]},
    "Uz": {"left": model_settings["Uz_left"], "right": model_settings["Uz_right"]},
}

# Initial temperature field (stored per node)
initial_temp = np.full((1, fe_mesh.GetNumberOfNodes()), 1e-4)

# Material parameters
material_dict = {
    "young_modulus": 1.0,
    "poisson_ratio": 0.3,
    "T0": initial_temp.flatten()
}

# Thermal parameters
thermal_dict = {"alpha": 1.5, "beta": 2.0, "c": 2.0}

# Create FE-based thermo-mechanical loss
thermomech_loss_3d = ThermoMechanicsLoss3DHexa(
    "thermomechanical_loss_3d",
    loss_settings={
        "dirichlet_bc_dict": bc_dict,
        "material_dict": material_dict,
        "thermal_dict": thermal_dict
    },
    fe_mesh=fe_mesh
)

thermomech_loss_3d.Initialize()

2026-02-13 14:26:14 - Info : thermomechanical_loss_3d.__init__ - finished in 0.0000 seconds
2026-02-13 14:26:15 - Info : thermomechanical_loss_3d.Initialize - element batch size is 50
2026-02-13 14:26:16 - Info : thermomechanical_loss_3d.Initialize - finished in 1.9465 seconds


## 5) Control definition

In the reference implementation this demo uses an **IdentityControl**, meaning:
- the "control variable" is passed through as-is (no transformation)
- here it typically means the input parameter field (e.g., `K`) is directly used as the control.

This is used by the FOL training wrapper.


In [4]:
from fol.controls.identity_control import IdentityControl
I_control = IdentityControl("I_Control", {}, fe_mesh.GetNumberOfNodes())
I_control.Initialize()

2026-02-13 14:26:30 - Info : I_Control.Initialize - finished in 0.0000 seconds


## Fourier‑parameterized sampling of the coefficient field $K$

A compact stochastic parameterization of spatially varying coefficients is obtained by expanding $K(x,y,z)$ in a truncated Fourier basis. For each prescribed set of modal frequencies:
1. Sample random Fourier coefficients,
2. Map them to a nodal field on the FE mesh,
3. Enforce value bounds $[K_{\min}, K_{\max}]$ through the control definition.

The resulting dataset `K_matrix` has shape:
$$
(\text{n\_samples},\; \text{n\_nodes})
$$
and is later reshaped to a structured grid for the FNO forward pass.


In [5]:
import jax
from fol.controls.fourier_control import FourierControl

freq_sets = [
    (np.array([2, 4, 6]), np.array([2, 4, 6]), np.array([2, 4, 6])),
    (np.array([1, 2, 3]), np.array([1, 2, 3]), np.array([1, 2, 3])),
    (np.array([3, 4, 5]), np.array([3, 4, 5]), np.array([3, 4, 5])),
    (np.array([4, 6, 8]), np.array([4, 6, 8]), np.array([4, 6, 8])),
]

K_matrix_parts = []

# PRNG handling: split keys so each frequency set yields distinct samples
seed = 42
key = jax.random.PRNGKey(seed)

N_samples_per_set = 1500

for idx, (x_freqs, y_freqs, z_freqs) in enumerate(freq_sets):
    key, subkey = jax.random.split(key)

    control_settings = {
        "x_freqs": x_freqs,
        "y_freqs": y_freqs,
        "z_freqs": z_freqs,
        "beta": 10,   # smoothness/sharpness parameter (implementation-dependent)
        "min": 0.1,   # lower bound for K
        "max": 1.0,   # upper bound for K
    }

    fourier_control = FourierControl("K", control_settings, fe_mesh)
    fourier_control.Initialize()

    # Sample random Fourier coefficients (one row per realization)
    coeffs = jax.random.normal(
        subkey, (N_samples_per_set, fourier_control.GetNumberOfVariables())
    )

    # Map coefficients -> bounded spatial field K on FE mesh nodes
    K_realizations = fourier_control.ComputeBatchControlledVariables(coeffs)

    K_matrix_parts.append(K_realizations)

# Concatenate all sets into a single dataset
K_matrix = jnp.concatenate(K_matrix_parts, axis=0)

print(f"{K_matrix.shape[0]} Fourier‑parameterized samples generated.")
print("K_matrix shape:", K_matrix.shape)  # (n_samples, n_nodes)

2026-02-13 14:26:37 - Info : K.Initialize - finished in 0.1817 seconds
2026-02-13 14:26:38 - Info : K.Initialize - finished in 0.0013 seconds
2026-02-13 14:26:38 - Info : K.Initialize - finished in 0.0011 seconds
2026-02-13 14:26:38 - Info : K.Initialize - finished in 0.0011 seconds
6000 Fourier‑parameterized samples generated.
K_matrix shape: (6000, 1331)


## Multi‑output Fourier Neural Operator (FNO)

A Fourier Neural Operator learns mappings between function spaces by combining:
- pointwise lifting/projection layers, and
- spectral convolution blocks operating on truncated Fourier modes.

Here, a **multi‑output** surrogate is constructed by instantiating one scalar‑output FNO per channel and concatenating predictions. The four output channels correspond to:
$$
(T,\;U_x,\;U_y,\;U_z).
$$
This design is convenient when the underlying FNO implementation is naturally scalar‑output while maintaining a shared input representation.


In [7]:
from fol.deep_neural_networks.ported_fourier_neural_operator_networks.fno import FNO
from flax import nnx

class MultiFNO(nnx.Module):
    def __init__(
        self,
        *,
        in_channels: int,
        out_channels: int = 4,
        hidden_channels: int = 16,
        n_modes=(10, 10, 10),
        n_layers: int = 4,
        rngs: nnx.Rngs,
    ):
        # Create `out_channels` independent FNOs, each predicting 1 channel
        self.fnos = nnx.List([
            FNO(
                in_channels=in_channels,
                out_channels=1,
                hidden_channels=hidden_channels,
                n_modes=n_modes,
                n_layers=n_layers,
                rngs=rngs,  # NNX will draw fresh keys internally
            )
            for _ in range(out_channels)
        ])

    def __call__(self, x: jnp.ndarray):
        # Run each FNO on the same input, concatenate outputs on channel axis
        outs = [fno(x) for fno in self.fnos]  # each (..., 1)
        return jnp.concatenate(outs, axis=-1)  # (..., out_channels)


### Instantiate the model and sanity-check shapes

the input `K_matrix` is stored as flattened node vectors, but FNOs usually expect a **grid**.
Since the mesh is structured `N x N x N` elements, the node grid is `(N+1)³`.

So we reshape from:
- `(B, num_nodes)` → `(B, N+1, N+1, N+1, 1)`


In [8]:
fno_model = MultiFNO(
    in_channels=1,
    out_channels=4,
    hidden_channels=16,
    n_modes=(10, 10, 10),
    n_layers=4,
    rngs=nnx.Rngs(0),
)

# Count trainable parameters
params = nnx.state(fno_model, nnx.Param)
total_params = sum(np.prod(x.shape) for x in jax.tree_util.tree_leaves(params))
print(f"FNO trainable parameters: {total_params}")

# Sanity-check forward pass with a small batch
B = 8
N = model_settings["N"]
grid_shape = (B, N+1, N+1, N+1, 1)
x0 = K_matrix[0:B].reshape(grid_shape)

y0 = fno_model(x0)
print("Input shape :", x0.shape)
print("Output shape:", y0.shape)  # expected (B, N+1, N+1, N+1, 4)


FNO trainable parameters: 2471748
Input shape : (8, 11, 11, 11, 1)
Output shape: (8, 11, 11, 11, 4)


## 8) Training setup (optimizer + schedule)

We use:
- a linear learning rate schedule from `1e-2` to `1e-3`
- Adam optimizer


In [9]:
import optax
num_epochs = 1000

learning_rate_scheduler = optax.linear_schedule(
    init_value=1e-2,
    end_value=1e-3,
    transition_steps=num_epochs
)

optimizer = optax.chain(optax.adam(learning_rate_scheduler))


## 9) Physics-Informed Fourier Parametric Operator Learning

This wrapper:
- takes the **control** (here identity control)
- the **physics loss** (thermo-mechanics)
- the **neural operator** (MultiFNO)
- and an **Optax optimizer**
then trains the network by minimizing physics residuals under the sampled `K` fields.

> In other words: the model learns an **operator** mapping `K → (T, Ux, Uy, Uz)` that satisfies the PDE.


In [10]:
from fol.deep_neural_networks.fourier_parametric_operator_learning import (
    PhysicsInformedFourierParametricOperatorLearning
)

pi_fno_pr_learning = PhysicsInformedFourierParametricOperatorLearning(
    name="pi_fno_pr_learning",
    control=I_control,
    loss_function=thermomech_loss_3d,
    flax_neural_network=fno_model,
    optax_optimizer=optimizer
)

pi_fno_pr_learning.Initialize()

2026-02-13 14:28:22 - Info : I_Control.Initialize - finished in 0.0000 seconds
2026-02-13 14:28:22 - Info : pi_fno_pr_learning.Initialize - finished in 0.4146 seconds


### Train / test split

we use the indices:
- Train: samples `[0:5000]`
- Test: samples `[5000:6000]`

> Make sure `K_matrix` has at least 6000 rows. With 4×1500 = 6000, it matches the reference implementation.


In [11]:
train_start_id, train_end_id = 0, 5000
test_start_id, test_end_id   = 5000, 6000

assert K_matrix.shape[0] >= test_end_id, "K_matrix doesn't have enough samples."

print("Train samples:", train_end_id - train_start_id)
print("Test samples :", test_end_id - test_start_id)


Train samples: 5000
Test samples : 1000


### Physics‑informed training

Training minimizes the thermo‑mechanical residual loss evaluated over mini‑batches of coefficient fields $K$. The test set is periodically evaluated to monitor generalization across unseen $K$ realizations.

Key settings:
- `test_frequency`: test evaluation interval (epochs)
- `batch_size`: number of coefficient fields per optimization step
- checkpointing: stores best‑loss state and periodic snapshots
- plotting: exports diagnostics at a fixed cadence


In [17]:
# pi_fno_pr_learning.Train(
#     train_set=(K_matrix[train_start_id:train_end_id, :],),
#     test_set=(K_matrix[test_start_id:test_end_id, :],),
#     test_frequency=100,
#     batch_size=50,
#     convergence_settings={
#         "num_epochs": num_epochs,
#         "relative_error": 1e-100,
#         "absolute_error": 1e-100,
#     },
#     train_checkpoint_settings={"least_loss_checkpointing": True, "frequency": 100},
#     plot_settings={"plot_save_rate": 100},
#     working_directory=case_dir
# )
display(Image(filename=os.path.join(case_dir,f'training_history.png')))

NameError: name 'Image' is not defined

### Restore the best saved model state

the reference implementation restores from `case_dir + "/flax_train_state"`.


In [13]:
pi_fno_pr_learning.RestoreState(restore_state_directory=case_dir + "/flax_final_state")

2026-02-13 14:43:47 - Info : pi_fno_pr_learning.RestoreState - flax nnx state is restored from ./pi_fno_thermo_mechanical_3d_box/flax_final_state


## Reference nonlinear finite‑element solve (validation)

To assess surrogate fidelity, the coupled problem is also solved using a **nonlinear residual‑based FE solver**. This provides reference solutions for selected $K$ realizations, enabling qualitative and quantitative comparison of:
- primary fields $(T,U_x,U_y,U_z)$,
- derived stress measures,
- and thermal flux fields.


In [14]:
from fol.solvers.fe_nonlinear_residual_based_solver import FiniteElementNonLinearResidualBasedSolver
fe_setting = {
    "linear_solver_settings": {
        "solver": "JAX-bicgstab",
        "tol": 1e-6,
        "atol": 1e-6,
        "maxiter": 1000,
        "pre-conditioner": "ilu",
    },
    "nonlinear_solver_settings": {
        "rel_tol": 1e-7,
        "abs_tol": 1e-7,
        "maxiter": 10,
        "load_incr": 5,
    }
}

nonlinear_fe_solver = FiniteElementNonLinearResidualBasedSolver(
    "nonlinear_fe_solver",
    thermomech_loss_3d,
    fe_setting
)
nonlinear_fe_solver.Initialize()

2026-02-13 14:44:10 - Info : nonlinear_fe_solver.__init__ - finished in 0.0000 seconds
2026-02-13 14:44:10 - Info : nonlinear_fe_solver.__init__ - finished in 0.0002 seconds
2026-02-13 14:44:10 - Info : nonlinear_fe_solver.Initialize - finished in 0.0000 seconds
2026-02-13 14:44:10 - Info : nonlinear_fe_solver.Initialize - finished in 0.0002 seconds


## 11) Compare PI-FNO vs FE on selected samples

We pick two sample indices (as in the reference implementation): `3000` and `5500`.

For each:
1. Store the input `K` field on the mesh
2. Predict `T, Ux, Uy, Uz` with PI-FNO
3. Compute derived quantities (stress, heat flux) from the predicted fields
4. Solve the FE problem and compute FE stress
5. Store everything in the mesh fields for export


In [15]:
for test_id in [3000, 5500]:

    # Store K field on the mesh (useful for visualization)
    fe_mesh[f'K_{test_id}'] = np.array(K_matrix[test_id, :]).reshape(-1, 1)

    # --- PI-FNO prediction ---
    FOL_TUVW = np.array(
        pi_fno_pr_learning.Predict(np.array(K_matrix[test_id, :]).reshape(1, -1))
    )  # shape (1, num_nodes*4) or similar depending on implementation

    # Store predicted primary fields
    fe_mesh[f'FOL_TUVW_{test_id}'] = FOL_TUVW.reshape((-1, 4))

    # Derived quantities from prediction
    fe_mesh[f'FOL_Stress_{test_id}'] = np.array(
        thermomech_loss_3d.ComputeStress(
            np.array(K_matrix[test_id, :]).flatten(),
            FOL_TUVW
        )
    )

    fe_mesh[f'FOL_Heat_Flux{test_id}'] = np.array(
        thermomech_loss_3d.ComputeHeatFlux(
            np.array(K_matrix[test_id, :]).flatten(),
            FOL_TUVW.reshape((-1, 4))[:, 0].flatten()  # temperature channel
        )
    )

    # --- FE reference solve ---
    # Initial guess for (T,Ux,Uy,Uz) stacked vector
    x0 = np.zeros((fe_mesh.GetNumberOfNodes() * 4,))

    FE_TUVW = np.array(
        nonlinear_fe_solver.Solve(np.array(K_matrix[test_id, :]).flatten(), x0)
    )

    fe_mesh[f'FE_TUVW_{test_id}'] = FE_TUVW.reshape((-1, 4))

    fe_mesh[f'FE_Stress_{test_id}'] = np.array(
        thermomech_loss_3d.ComputeStress(
            np.array(K_matrix[test_id, :]).flatten(),
            FE_TUVW.reshape((-1, 4))[:, 0].flatten()
        )
    )

    print(f"Done sample {test_id}: stored PI-FNO + FE results on mesh.")


2026-02-13 14:44:44 - Info : pi_fno_pr_learning.Predict - finished in 2.6543 seconds
2026-02-13 14:44:45 - Info : thermomechanical_loss_3d.ComputeStress - finished in 0.8852 seconds
2026-02-13 14:44:45 - Info : thermomechanical_loss_3d.ComputeHeatFlux - finished in 0.5239 seconds
2026-02-13 14:44:45 - Info : thermomechanical_loss_3d.ApplyDirichletBCOnDofVector - finished in 0.0383 seconds
2026-02-13 14:44:47 - Info : thermomechanical_loss_3d.ComputeJacobianMatrixAndResidualVector - finished in 1.9251 seconds
2026-02-13 14:44:48 - Info : nonlinear_fe_solver.JaxBicgstabLinearSolver - finished in 0.5637 seconds
2026-02-13 14:44:48 - Info : nonlinear_fe_solver.Solve - 
      ───────── Load Step 1 ─────────
        Newton Iteration : 1 (max = 10)
        Residual Norm    : 4.674e-02 (abs_tol = 1.000e-07)
        Δ DOFs Norm      : 2.268e+00 (rel_tol = 1.000e-07)
        Converged        : False
     ────────────────────────────────────────────
2026-02-13 14:44:48 - Info : thermomechanical_l

## 12) Export results

Finally, we finalize the mesh and export all stored fields to `case_dir`.
You can open the exported results in the usual post-processing pipeline.


In [16]:
fe_mesh.Finalize(export_dir=case_dir)

2026-02-13 14:45:20 - Info : box_io.Finalize - finished in 0.0016 seconds
